# 1. Collecting the Data

Everything the model needs, downloaded once into `data/`:

1. **The SEC Financial Statement Data Sets**, 2009 Q1 to 2026 Q3 (71 quarters, about 7 GB), in `data/zips/`. These have the numbers of every 10-K and 10-Q.
2. **SEC full text search**, one month at a time from 2007 to 2026, in `data/efts/<search>/<YYYY-MM>.json`:
   - the bankruptcy 8-Ks (Item 1.03), for the labels
   - the late filing notices (NT 10-K, NT 10-Q)
   - the 10-Ks and 10-Qs with warning phrases like "substantial doubt" + "going concern" or "forbearance agreement"

Files that are already there are skipped, so it can be stopped and started again.

The SEC asks for a name and email in every request. Set it before starting Jupyter:
`export SEC_USER_AGENT="Your Name your.email@school.edu"`

In [1]:
import json
import os
import time

import pandas as pd
import requests


def sec_headers():
    #the SEC wants a name and email with every request, set in the SEC_USER_AGENT environment variable
    if not os.environ.get("SEC_USER_AGENT"):
        raise RuntimeError("set SEC_USER_AGENT to your name and email first")
    return {"User-Agent": os.environ["SEC_USER_AGENT"]}


os.makedirs("data/zips", exist_ok=True)
os.makedirs("data/efts", exist_ok=True)

/Users/pavan/Desktop/Capstone/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Financial Statement Data Sets

In [2]:
QUARTERS = [f"{y}q{q}" for y in range(2009, 2027) for q in range(1, 5)][:71]

for quarter in QUARTERS:
    path = f"data/zips/{quarter}.zip"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        continue
    r = requests.get(f"https://www.sec.gov/files/dera/data/financial-statement-data-sets/{quarter}.zip", headers=sec_headers(), timeout=600)
    r.raise_for_status()
    open(path, "wb").write(r.content)
    print("got", quarter, round(len(r.content) / 1e6), "MB")
    time.sleep(1)
print(len(os.listdir("data/zips")), "zip files")

71 zip files


## Full Text Search

Each search is run for one month at a time (the search gives at most 10,000 results per request). A filing can be for a parent company and its subsidiaries together, so all of its ciks are kept.

In [ ]:
URL = "https://efts.sec.gov/LATEST/search-index"
SEARCHES = {
    #the label: bankruptcy or receivership
    "item_1.03": {"q": '"Item 1.03"', "forms": "8-K"},
    #late filing notices
    "nt_filing": {"q": None, "forms": "NT 10-K,NT 10-Q"},
    #phrases in the 10-K and 10-Q text
    "going_concern": {"q": '"substantial doubt" "going concern"', "forms": "10-K,10-Q"},
    "forbearance": {"q": '"forbearance agreement"', "forms": "10-K,10-Q"},
    "minimum_bid_price": {"q": '"minimum bid price"', "forms": "10-K,10-Q"},
    "chapter_11": {"q": '"chapter 11"', "forms": "10-K,10-Q"},
    "strategic_alternatives": {"q": '"strategic alternatives"', "forms": "10-K,10-Q"},
    "material_weakness": {"q": '"material weakness"', "forms": "10-K,10-Q"},
    "not_in_compliance": {"q": '"not in compliance"', "forms": "10-K,10-Q"},
    "reverse_stock_split": {"q": '"reverse stock split"', "forms": "10-K,10-Q"},
    "restructuring_support_agreement": {"q": '"restructuring support agreement"', "forms": "10-K,10-Q"},
    #stricter phrases: "material weakness" alone also finds "we did not identify any material weakness"
    "identified_material_weakness": {"q": '"identified a material weakness"', "forms": "10-K,10-Q"},
    "covenant_not_in_compliance": {"q": '"not in compliance with" "covenants"', "forms": "10-K,10-Q"},
    "notice_of_default": {"q": '"notice of default"', "forms": "10-K,10-Q"},
    "debtor_in_possession": {"q": '"debtor-in-possession"', "forms": "10-K,10-Q"},
    "cease_operations": {"q": '"cease operations"', "forms": "10-K,10-Q"},
    "wind_down": {"q": '"wind down"', "forms": "10-K,10-Q"},
}


def get(params):
    #the search sometimes fails for a few minutes, so it waits longer each time
    for attempt in range(12):
        try:
            r = requests.get(URL, params=params, headers=sec_headers(), timeout=60)
            #sometimes the SEC answers 200 with an error message instead of results, so that is retried too
            if r.status_code == 200 and "hits" in r.json():
                return r.json()["hits"]
        except (requests.RequestException, ValueError):
            pass
        time.sleep(5 + 10 * attempt)
    raise RuntimeError(f"failed: {params}")


def month_hits(search, start, end):
    params = {"forms": search["forms"], "startdt": start, "enddt": end, "dateRange": "custom"}
    if search["q"]:
        params["q"] = search["q"]
    rows, position = {}, 0
    while True:
        hits = get({**params, "from": position})
        for h in hits["hits"]:
            s = h["_source"]
            rows[s["adsh"]] = {"adsh": s["adsh"], "ciks": [int(c) for c in s["ciks"]], "form": s["form"],
                               "file_date": s["file_date"], "items": s.get("items", [])}
        position += len(hits["hits"])
        if len(hits["hits"]) == 0 or position >= hits["total"]["value"] or position >= 10000:
            if position >= 10000:
                print("  over 10000 hits, some missing:", start)
            return list(rows.values())
        time.sleep(0.12)


months = pd.date_range("2007-01-01", "2026-10-31", freq="MS")
for name, search in SEARCHES.items():
    os.makedirs(f"data/efts/{name}", exist_ok=True)
    count = 0
    for m in months:
        path = f"data/efts/{name}/{m.strftime('%Y-%m')}.json"
        if os.path.exists(path):
            continue
        rows = month_hits(search, m.strftime("%Y-%m-%d"), (m + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d"))
        json.dump(rows, open(path, "w"))
        count += len(rows)
    print(f"{name}: {len(os.listdir(f'data/efts/{name}'))} months, {count} new filings")